# Required function

### 1. Neural signal processing

In [ ]:
import numpy as np

def meanSubtract(dat, area='6v'):
    """Subtracts the mean of each block from the neural features."""
    if area == '6v':
        dat['feat'] = np.concatenate([dat['tx2'][:, :128], dat['spikePow'][:, :128]], axis=1).astype(np.float32)
    elif area == '44':
        dat['feat'] = np.concatenate([dat['tx2'][:, 128:], dat['spikePow'][:, 128:]], axis=1).astype(np.float32)

    for b in np.unique(dat['blockNum']):
        idx = dat['blockNum'].squeeze() == b
        dat['feat'][idx] -= np.mean(dat['feat'][idx], axis=0, keepdims=True)

    return dat

### 2. Generate audio and mel-spec

In [ ]:
import os, warnings
import subprocess

# Prepend conda bin to PATH
conda_bin = "/home/mega/anaconda3/envs/envi/bin"
os.environ["PATH"] = conda_bin + os.pathsep + os.environ.get("PATH", "")

# Suppress pydub warnings
warnings.filterwarnings("ignore", message=".*ffprobe.*", category=RuntimeWarning)

# Verify both work
subprocess.run(["ffmpeg", "-version"], check=True)
subprocess.run(["ffprobe", "-version"], check=True)
print("✅ ffmpeg + ffprobe accessible!")

In [ ]:
import gtts
import tempfile
import librosa
import os
import torch
import numpy as np
from pydub import AudioSegment


def generate_audio(text):
    tts = gtts.gTTS(text=text, lang='en')
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as mp3_file:
        mp3_path = mp3_file.name
        tts.save(mp3_path)
        
    wav_path = mp3_path.replace('.mp3', '.wav')
    AudioSegment.from_mp3(mp3_path).export(wav_path, format='wav')
    
    audio_data, sample_rate = librosa.load(wav_path, sr=None)
    
    os.unlink(mp3_path)
    os.unlink(wav_path)
    return audio_data, sample_rate


def normalize_volume(audio):
    rms = librosa.feature.rms(y=audio)
    max_rms = rms.max() + 0.01
    target_rms = 0.2
    audio = audio * (target_rms/max_rms)
    max_val = np.abs(audio).max()
    if max_val > 1.0: 
        audio = audio / max_val
    return audio


def dynamic_range_compression_torch(x, C=1, clip_val=1e-5):
    return torch.log(torch.clamp(x, min=clip_val) * C)


def spectral_normalize_torch(magnitudes):
    output = dynamic_range_compression_torch(magnitudes)
    return output


mel_basis = {}
hann_window = {}
def mel_spectrogram(y, n_fft, num_mels, sampling_rate, hop_size, win_size, fmin, fmax, center=False):
    if torch.min(y) < -1.:
        print('min value is ', torch.min(y))
    if torch.max(y) > 1.:
        print('max value is ', torch.max(y))

    global mel_basis, hann_window
    if fmax not in mel_basis:
        mel = librosa.filters.mel(sr=sampling_rate, n_fft=n_fft, n_mels=num_mels, fmin=fmin, fmax=fmax)
        mel_basis[str(fmax)+'_'+str(y.device)] = torch.from_numpy(mel).float().to(y.device)
        hann_window[str(y.device)] = torch.hann_window(win_size).to(y.device)

    y = torch.nn.functional.pad(y.unsqueeze(1), (int((n_fft-hop_size)/2), int((n_fft-hop_size)/2)), mode='reflect')
    y = y.squeeze(1)

    spec = torch.stft(y, n_fft, hop_length=hop_size, win_length=win_size, window=hann_window[str(y.device)],
                      center=center, pad_mode='reflect', normalized=False, onesided=True, return_complex=True)
    spec = torch.view_as_real(spec)
    spec = torch.sqrt(spec.pow(2).sum(-1)+(1e-9))

    spec = torch.matmul(mel_basis[str(fmax)+'_'+str(y.device)], spec)
    spec = spectral_normalize_torch(spec)
    return spec


def load_audio(text, start=None, end=None, max_frames=None, renormalize_volume=False):
    audio, r = generate_audio(text)

    if len(audio.shape) > 1:
        audio = audio[:,0] 
    if start is not None or end is not None:
        audio = audio[start:end]

    if renormalize_volume:
        audio = normalize_volume(audio)

    if r != 22050:
        audio = librosa.resample(audio, orig_sr=r, target_sr=22050)
        r = 22050
    
    audio = np.clip(audio, -1, 1) 
    pytorch_mspec = mel_spectrogram(torch.tensor(audio, dtype=torch.float32).unsqueeze(0), 1024, 80, 22050, 256, 1024, 0, 8000, center=False)
    mspec = pytorch_mspec.squeeze(0).T.numpy()
    if max_frames is not None and mspec.shape[0] > max_frames:
        mspec = mspec[:max_frames,:]
    return mspec, audio

### 3. Align ecog and mel-spec

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances
from matplotlib import pyplot as plt


def time_warp(costs):
    dtw = np.zeros_like(costs)
    dtw[0,1:] = np.inf
    dtw[1:,0] = np.inf
    eps = 1e-4
    for i in range(1,costs.shape[0]):
        for j in range(1,costs.shape[1]):
            dtw[i,j] = costs[i,j] + min(dtw[i-1,j],dtw[i,j-1],dtw[i-1,j-1])
    return dtw


def compute_euclidean_distance(ecog, mspec, variance_threshold=0.95, n_components=None):
    if n_components is None:
        # Determine components to retain variance_threshold
        pca_ecog_full = PCA()
        pca_ecog_full.fit(ecog)
        cumsum_ecog = np.cumsum(pca_ecog_full.explained_variance_ratio_)
        n_comp_ecog = np.argmax(cumsum_ecog >= variance_threshold) + 1
        
        pca_mspec_full = PCA()
        pca_mspec_full.fit(mspec)
        cumsum_mspec = np.cumsum(pca_mspec_full.explained_variance_ratio_)
        n_comp_mspec = np.argmax(cumsum_mspec >= variance_threshold) + 1
    else:
        n_comp_ecog = min(n_components, ecog.shape[1])
        n_comp_mspec = min(n_components, mspec.shape[1])
    
    n_components_final = min(n_comp_ecog, n_comp_mspec)
    
    pca_ecog = PCA(n_components=n_components_final)
    pca_mspec = PCA(n_components=n_components_final)
    
    ecog_reduced = pca_ecog.fit_transform(ecog)
    mspec_reduced = pca_mspec.fit_transform(mspec)
    
    distance_matrix = euclidean_distances(ecog_reduced, mspec_reduced)
    return distance_matrix


def align_from_distances(ecog, mspec, debug=False):
    distance_matrix = compute_euclidean_distance(ecog, mspec, variance_threshold=0.95, n_components=None)
    dtw = time_warp(distance_matrix)

    i = distance_matrix.shape[0]-1
    j = distance_matrix.shape[1]-1
    results = [0] * distance_matrix.shape[0]
    while i > 0 and j > 0:
        results[i] = j
        i, j = min([(i-1,j),(i,j-1),(i-1,j-1)], key=lambda x: dtw[x[0],x[1]])

    if debug:
        visual = np.zeros_like(dtw)
        visual[range(len(results)),results] = 1
        plt.matshow(visual)
        plt.show()

    return results

### 4. Prepare data 

In [ ]:
import scipy.io as sio
from tqdm.auto import tqdm 

def load_data(filename, area='6v'):
    """Loads and processes ECoG and spectrogram data from MATLAB file."""
    data = sio.loadmat(filename)
    data = meanSubtract(data, area)

    goTrialEpochs = data['goTrialEpochs'] - 1
    cueList = data['cueList'][0, 1:51]
    trialCues = data['trialCues'] - 1
    
    ecog_data = []
    valid_trialCues = []

    # Extract segments based on goTrialEpochs and trialCues
    for idx, (start, end) in enumerate(goTrialEpochs):
        if trialCues[idx] != 0: # except do_nothing
            segment = data['feat'][start:end, :]
            ecog_data.append(segment)
            valid_trialCues.append(trialCues[idx] - 1) 
    trialCues = np.array(valid_trialCues).flatten()

    # Padding
    max_len = max(segment.shape[0] for segment in ecog_data)
    n_trials = len(ecog_data)
    n_channels = ecog_data[0].shape[1]

    ecog_data_padded = np.zeros((n_trials, max_len, n_channels))
    for i, segment in enumerate(ecog_data):
        length = segment.shape[0]
        ecog_data_padded[i, :length, :] = segment
    ecog_data = np.array(ecog_data_padded, dtype=np.float32)

    # Create map between label and text
    label_text_dict = {}
    for i in range(len(trialCues)):
        label = trialCues[i]
        label_text_dict[label] = cueList[label]

    fixedlabels = []
    for x in range(len(cueList)):
        fixedlabels.append(cueList[x])
    
    # Load mel spectrograms and audio
    all_mspec, all_audio = [], []
    for i in tqdm(range(len(fixedlabels))):
        text = fixedlabels[i][0]
        mspec, audio = load_audio(text)
        all_mspec.append(mspec)
        all_audio.append(audio)
    
    # Make dictionary
    label_text_spectrogram_dict = {}
    for label in label_text_dict:
        label_text_spectrogram_dict[label] = {
            "text": label_text_dict[label],
            "mspec": all_mspec[label],
        }
    sub_dict = [label_text_spectrogram_dict[label] for label in trialCues]
    
    texts = [item['text'][0] if isinstance(item['text'], (np.ndarray, list)) else item['text'] for item in sub_dict]
    mspecs = [item['mspec'] for item in sub_dict]
    
    dictionary = {
        'text': texts,
        'mspec': mspecs,
        'ecog': ecog_data,
        'label': trialCues
    }
    return dictionary

In [ ]:
from sklearn.model_selection import train_test_split

def prepare_dataloader_custom(ecog, mspec, labels, test_size=0.1, val_size=0.1, exclude_labels=None):
    """
    Custom dataloader preparation that excludes certain labels from training and validation sets
    and adds them to the test set
    """
    if exclude_labels is not None:
        # Separate excluded labels from the rest
        exclude_mask = np.isin(labels, exclude_labels)
        include_mask = ~exclude_mask
        
        # Data with excluded labels (goes to test set)
        ecog_excluded = [ecog[i] for i in range(len(ecog)) if exclude_mask[i]]
        mspec_excluded = [mspec[i] for i in range(len(mspec)) if exclude_mask[i]]
        labels_excluded = labels[exclude_mask]
        
        # Data without excluded labels (for train/val split)
        ecog_included = [ecog[i] for i in range(len(ecog)) if include_mask[i]]
        mspec_included = np.array([mspec[i] for i in range(len(mspec)) if include_mask[i]])
        labels_included = labels[include_mask]
        
        # Split the remaining data into train and validation
        if len(ecog_included) > 0:
            X_train, X_val, y_train, y_val, label_train, label_val = train_test_split(
                ecog_included, mspec_included, labels_included, 
                test_size=val_size, random_state=42, stratify=labels_included
            )
        else:
            X_train, X_val, y_train, y_val = [], [], np.array([]), np.array([])
            label_train, label_val = np.array([]), np.array([])
        
        # Combine excluded samples with any additional test samples
        if test_size > 0 and len(ecog_included) > 0:
            # Take additional samples for test set from the remaining data
            X_temp, X_test_extra, y_temp, y_test_extra, label_temp, label_test_extra = train_test_split(
                X_train, y_train, label_train, 
                test_size=test_size, random_state=42, stratify=label_train
            )
            # Update train set
            X_train, y_train, label_train = X_temp, y_temp, label_temp
            
            # Combine excluded samples with extra test samples
            X_test = ecog_excluded + list(X_test_extra)
            y_test = np.concatenate([mspec_excluded, y_test_extra])
            label_test = np.concatenate([labels_excluded, label_test_extra])
        else:
            # Only excluded samples go to test set
            X_test = ecog_excluded
            y_test = np.array(mspec_excluded)
            label_test = labels_excluded
    
    else:
        # Original behavior when no labels are excluded
        X_temp, X_test, y_temp, y_test, label_temp, label_test = train_test_split(
            ecog, mspec, labels, test_size=test_size, random_state=42, stratify=labels
        )
        
        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val, label_train, label_val = train_test_split(
            X_temp, y_temp, label_temp, test_size=val_size_adjusted, random_state=42, stratify=label_temp
        )
    
    # Convert lists to numpy arrays
    X_train = np.array(X_train) if len(X_train) > 0 else np.array([])
    X_val = np.array(X_val) if len(X_val) > 0 else np.array([])
    X_test = np.array(X_test) if len(X_test) > 0 else np.array([])

    return X_train, X_val, X_test, y_train, y_val, y_test, label_train, label_val, label_test

### 5. Diffusion model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import math


# -------------------------
# Noise schedules
# -------------------------
def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi / 2) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clamp(betas, 1e-8, 0.999)

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ECoGBlock(nn.Module):  
    def __init__(self, dim, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, dim)
        self.conv1 = nn.Conv1d(dim, dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(dim, dim, kernel_size=3, padding=1)
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(dim, max(dim // 4, 1), 1),
            nn.ReLU(),
            nn.Conv1d(max(dim // 4, 1), dim, 1),
            nn.Sigmoid()
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x, time_emb):
        h = x.transpose(1, 2)
        h = self.norm1(h)
        h = h.transpose(1, 2)
        h = F.relu(h)
        h = self.conv1(h)

        time_emb_proj = self.time_mlp(time_emb)
        h = h + time_emb_proj.unsqueeze(-1)

        h = h.transpose(1, 2)
        h = self.norm2(h)
        h = h.transpose(1, 2)
        h = F.relu(h)
        h = self.dropout(h)
        h = self.conv2(h)

        attention = self.channel_attention(h)
        h = h * attention

        return x + h  # Residual

class Downsample1d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)

class Upsample1d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.tconv = nn.ConvTranspose1d(
            in_ch, out_ch, kernel_size=4, stride=2, padding=1, output_padding=0
        )

    def forward(self, x):
        return self.tconv(x)

def match_length(x, target_len: int):
    L = x.shape[-1]
    if L == target_len:
        return x
    if L > target_len:
        start = (L - target_len) // 2
        return x[..., start:start + target_len]
    pad = target_len - L
    return F.pad(x, (0, pad))

class ECoGDiffusionModel(nn.Module):
    def __init__(
        self,
        channels=256,
        seq_len=101,
        base_dim=128,
        dim_mults=(1, 2, 4),
        time_emb_dim=256,
    ):
        super().__init__()

        self.channels = channels
        self.seq_len = seq_len

        self.dims = dims = [base_dim * m for m in dim_mults]
        self.mid_dim = dims[-1] * 2

        self.time_embedding = SinusoidalPositionEmbeddings(time_emb_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.ReLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )

        self.input_proj = nn.Conv1d(channels, dims[0], kernel_size=1)

        self.enc_blocks = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        for i, dim in enumerate(dims):
            self.enc_blocks.append(ECoGBlock(dim, time_emb_dim))
            out_dim = dims[i + 1] if i < len(dims) - 1 else self.mid_dim
            self.downsamples.append(Downsample1d(dim, out_dim))

        self.mid_block = ECoGBlock(self.mid_dim, time_emb_dim)

        self.upsamples = nn.ModuleList()
        self.fuse = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()

        decoder_dims = [self.mid_dim] + dims[::-1]
        for i in range(len(decoder_dims) - 1):
            in_dim = decoder_dims[i]
            out_dim = decoder_dims[i + 1]
            self.upsamples.append(Upsample1d(in_dim, out_dim))
            self.fuse.append(nn.Conv1d(out_dim * 2, out_dim, kernel_size=1))
            self.dec_blocks.append(ECoGBlock(out_dim, time_emb_dim))

        self.output_proj = nn.Conv1d(dims[0], channels, kernel_size=1)

    def forward(self, x, t, label_emb=None):
        time_emb = self.time_mlp(self.time_embedding(t))
        if label_emb is not None:
            time_emb = time_emb + label_emb

        h = self.input_proj(x)

        skips = []
        for block, down in zip(self.enc_blocks, self.downsamples):
            h = block(h, time_emb)
            skips.append(h)
            h = down(h)

        h = self.mid_block(h, time_emb)

        for up, fuse, block in zip(self.upsamples, self.fuse, self.dec_blocks):
            h = up(h)
            skip = skips.pop()
            h = match_length(h, skip.shape[-1])
            h = torch.cat([h, skip], dim=1)
            h = fuse(h)
            h = block(h, time_emb)

        out = self.output_proj(h)
        out = match_length(out, self.seq_len)
        return out

class ECoGDiffusion:
    def __init__(
        self,
        model,
        timesteps=1000,
        beta_start=1e-4,
        beta_end=0.02,
        schedule="linear",
    ):
        self.model = model
        self.timesteps = timesteps

        if schedule == "linear":
            self.betas = linear_beta_schedule(timesteps, beta_start, beta_end)
        elif schedule == "cosine":
            self.betas = cosine_beta_schedule(timesteps)
        else:
            raise ValueError(f"Unknown schedule: {schedule}")

        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

        eps = 1e-8
        self.snr = self.alphas_cumprod / (1.0 - self.alphas_cumprod + eps)
        self.log_snr = torch.log(self.snr)

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)

        alphas_cumprod_t = self.alphas_cumprod[t].view(-1, 1, 1)
        sqrt_alphas_cumprod_t = torch.sqrt(alphas_cumprod_t)
        sqrt_one_minus_alphas_cumprod_t = torch.sqrt(1 - alphas_cumprod_t)

        return sqrt_alphas_cumprod_t * x0 + sqrt_one_minus_alphas_cumprod_t * noise
    
    def sample_timesteps(self, batch_size, device):
        return torch.randint(0, self.timesteps, (batch_size,), device=device)
    
    def training_step(self, x0):
        device = x0.device
        batch_size = x0.shape[0]

        t = self.sample_timesteps(batch_size, device)
        noise = torch.randn_like(x0)

        xt = self.add_noise(x0, t, noise)

        predicted_noise = self.model(xt, t)

        loss = F.mse_loss(predicted_noise, noise)
        return loss
    
    @torch.no_grad()
    def sample(self, batch_size, device, condition=None):
        self.model.eval()

        x = torch.randn(batch_size, self.model.channels, self.model.seq_len, device=device)

        for t in tqdm(reversed(range(self.timesteps)), desc="Sampling"):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
            predicted_noise = self.model(x, t_batch)

            alpha_t = self.alphas[t]
            alpha_cumprod_t = self.alphas_cumprod[t]
            beta_t = self.betas[t]

            if t > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)

            x = (1 / torch.sqrt(alpha_t)) * (
                x - (beta_t / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise
            ) + torch.sqrt(beta_t) * noise
        
        return x

class ConditionalECoGDiffusion(ECoGDiffusion):
    def __init__(
        self,
        model,
        num_labels,
        timesteps=1000,
        beta_start=1e-4,
        beta_end=0.02,
        schedule="linear",
    ):
        super().__init__(model, timesteps, beta_start, beta_end, schedule)
        self.num_labels = num_labels
        self.label_embedding = nn.Embedding(num_labels, model.time_mlp[0].in_features)
        
    def training_step(self, x0, labels):
        device = x0.device
        batch_size = x0.shape[0]

        t = self.sample_timesteps(batch_size, device)
        noise = torch.randn_like(x0)

        xt = self.add_noise(x0, t, noise)

        label_emb = self.label_embedding(labels)

        predicted_noise = self.model(xt, t, label_emb)

        loss = F.mse_loss(predicted_noise, noise)
        return loss
    
    @torch.no_grad()
    def sample_conditional(self, labels, device):
        self.model.eval()
        batch_size = len(labels)

        x = torch.randn(batch_size, self.model.channels, self.model.seq_len, device=device)

        label_emb = self.label_embedding(labels)

        for t in tqdm(reversed(range(self.timesteps))):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)

            predicted_noise = self.model(x, t_batch, label_emb)

            alpha_t = self.alphas[t]
            alpha_cumprod_t = self.alphas_cumprod[t]
            beta_t = self.betas[t]

            if t > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)

            x = (1 / torch.sqrt(alpha_t)) * (
                x - (beta_t / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise
            ) + torch.sqrt(beta_t) * noise
        
        return x

In [ ]:
def train_diffusion_model(
    ecog,
    label,
    num_epochs,
    batch_size,
    learning_rate,
    device,
    schedule="linear",  # "linear" or "cosine"
):
    if isinstance(ecog, list):
        ecog = np.array(ecog)
    if isinstance(label, list):
        label = np.array(label)
    
    unique_labels = np.unique(label)
    label_mapping = {old_label: new_label for new_label, old_label in enumerate(unique_labels)}
    reverse_mapping = {new_label: old_label for old_label, new_label in label_mapping.items()}
    
    remapped_labels = np.array([label_mapping[l] for l in label])
    
    channels = ecog.shape[-1]
    seq_len = ecog.shape[1]
    num_labels = len(unique_labels)

    model = ECoGDiffusionModel(channels=channels, seq_len=seq_len).to(device)

    diffusion = ConditionalECoGDiffusion(
        model=model,
        num_labels=num_labels,
        timesteps=1000,
        schedule=schedule,
    )
    
    diffusion.label_mapping = label_mapping
    diffusion.reverse_mapping = reverse_mapping
    
    diffusion.betas = diffusion.betas.to(device)
    diffusion.alphas = diffusion.alphas.to(device)
    diffusion.alphas_cumprod = diffusion.alphas_cumprod.to(device)
    diffusion.snr = diffusion.snr.to(device)
    diffusion.log_snr = diffusion.log_snr.to(device)
    diffusion.label_embedding = diffusion.label_embedding.to(device)

    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(diffusion.label_embedding.parameters()), 
        lr=learning_rate
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_epochs)

    dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(ecog).transpose(1, 2),
        torch.LongTensor(remapped_labels)
    )
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model.train()
    diffusion.label_embedding.train()
    
    epoch_loss = []
    
    for epoch in range(num_epochs):
        total_loss = 0
        valid_batches = 0
        
        for batch_idx, (x0, batch_labels) in enumerate(dataloader):
            x0, batch_labels = x0.to(device), batch_labels.to(device)
            optimizer.zero_grad()
            loss = diffusion.training_step(x0, batch_labels)
            
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Invalid loss at epoch {epoch}, batch {batch_idx}: {loss}")
                continue
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(diffusion.label_embedding.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            valid_batches += 1

        scheduler.step()
        avg_loss = total_loss / max(valid_batches, 1)
        epoch_loss.append(avg_loss)
        
        if (epoch + 1) % 10 == 0 or (epoch + 1) == num_epochs:
            print(f"[{schedule}] Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.6f}")

    return model, diffusion, epoch_loss

In [ ]:
def generate_synthetic_ecog(model, diffusion, original_labels, num_samples_per_label=5):
    """Generate synthetic ECoG data with proper label handling"""
    device = next(model.parameters()).device
    synthetic_data = []
    synthetic_labels = []
    
    unique_labels = np.unique(original_labels)
    
    # Ensure diffusion components are on correct device
    diffusion.betas = diffusion.betas.to(device)
    diffusion.alphas = diffusion.alphas.to(device)
    diffusion.alphas_cumprod = diffusion.alphas_cumprod.to(device)
    diffusion.label_embedding = diffusion.label_embedding.to(device)
    
    model.eval()
    diffusion.label_embedding.eval()

    for original_label in unique_labels:
        # Map to training label space
        if hasattr(diffusion, 'label_mapping') and original_label in diffusion.label_mapping:
            mapped_label = diffusion.label_mapping[original_label]
        else:
            # Fallback: find index in unique labels
            mapped_label = np.where(unique_labels == original_label)[0][0]
        
        # Verify mapped label is valid
        if mapped_label >= diffusion.label_embedding.num_embeddings:
            print(f"ERROR: Mapped label {mapped_label} exceeds embedding size {diffusion.label_embedding.num_embeddings}")
            continue
        
        # Generate in small batches to avoid memory issues
        batch_size = min(num_samples_per_label, 3)
        for start_idx in range(0, num_samples_per_label, batch_size):
            current_batch_size = min(batch_size, num_samples_per_label - start_idx)
            labels_batch = torch.full((current_batch_size,), mapped_label, device=device, dtype=torch.long)
            
            try:
                with torch.no_grad():
                    generated = diffusion.sample_conditional(labels_batch, device)
                    generated = generated.transpose(1, 2).cpu().numpy()  # Back to [batch, seq_len, channels]
                
                synthetic_data.extend(generated)
                # Use original labels for consistency with training data
                synthetic_labels.extend([original_label] * current_batch_size)
                
            except RuntimeError as e:
                print(f"Error generating for label {original_label}: {e}")
                continue
    
    print(f"Generated {len(synthetic_data)} synthetic samples")
    return np.array(synthetic_data), np.array(synthetic_labels)

### 6. Feature extraction

In [ ]:
class ConvLSTMFeatureExtractor(nn.Module):
    def __init__(self, input_size=256, conv_channels=64, lstm_hidden=128, 
                 output_size=80, dropout=0.2):
        super(ConvLSTMFeatureExtractor, self).__init__()
        
        # Multi-scale convolutional feature extraction
        self.conv_block1 = self._make_conv_block(input_size, conv_channels, 3)
        self.conv_block2 = self._make_conv_block(conv_channels, conv_channels*2, 5)
        self.conv_block3 = self._make_conv_block(conv_channels*2, conv_channels*2, 7)
        
        # Attention for spatial (channel) selection
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(conv_channels*2, conv_channels//2, 1),
            nn.ReLU(),
            nn.Conv1d(conv_channels//2, conv_channels*2, 1),
            nn.Sigmoid()
        )
        
        # Bidirectional LSTM for temporal modeling
        self.lstm = nn.LSTM(
            input_size=conv_channels*2,
            hidden_size=lstm_hidden,
            num_layers=2,
            dropout=dropout,
            bidirectional=True,
            batch_first=True
        )
        
        # Self-attention for temporal relationships
        lstm_output_size = lstm_hidden * 2
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=lstm_output_size,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Output projection with residual
        self.output_proj = nn.Sequential(
            nn.Linear(lstm_output_size, lstm_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, output_size)
        )
        
        # Residual connection
        self.residual = nn.Linear(input_size, output_size)
        self.layer_norm = nn.LayerNorm(output_size)
        
    def _make_conv_block(self, in_channels, out_channels, kernel_size):
        return nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        
    def forward(self, x):
        batch_size, seq_len, input_size = x.shape
        residual = self.residual(x)
        
        # Convolutional feature extraction
        x_conv = x.transpose(1, 2)  # [batch, input_size, seq_len]
        conv1 = self.conv_block1(x_conv)
        conv2 = self.conv_block2(conv1)
        conv3 = self.conv_block3(conv2)
        
        # Apply channel attention
        attention = self.channel_attention(conv3)
        conv_features = conv3 * attention
        conv_features = conv_features.transpose(1, 2)  # [batch, seq_len, conv_channels*2]
        
        # LSTM processing
        lstm_out, _ = self.lstm(conv_features)
        
        # Temporal attention
        attended_out, _ = self.temporal_attention(lstm_out, lstm_out, lstm_out)
        attended_out = lstm_out + attended_out  # Residual connection
        
        # Output projection
        output = self.output_proj(attended_out)
        output = output + residual
        output = self.layer_norm(output)
        
        return output

### 7. MoETransformer

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


class Expert(nn.Module):
    """Individual expert network in MoE architecture"""
    
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through expert network"""
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class GatingNetwork(nn.Module):
    """Gating network that routes inputs to appropriate experts"""
    
    def __init__(self, input_dim: int, num_experts: int, top_k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(input_dim, num_experts)
        
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        gate_logits = self.gate(x)  # [N, num_experts]

        # 🔹 STEP 2: add exploration noise (TRAINING ONLY)
        if self.training:
            gate_logits = gate_logits + torch.randn_like(gate_logits) * 0.1

        # Top-k routing
        top_k_gates, top_k_indices = torch.topk(gate_logits, self.top_k, dim=-1)

        # Normalize gates
        top_k_gates = F.softmax(top_k_gates, dim=-1)

        return top_k_gates, top_k_indices


class MixtureOfExperts(nn.Module):
    """Complete Mixture of Experts layer with load balancing"""
    
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        num_experts: int = 8,
        top_k: int = 2,
        load_balance_weight: float = 0.01
    ):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.load_balance_weight = load_balance_weight
        
        # Create expert networks
        self.experts = nn.ModuleList([
            Expert(input_dim, hidden_dim, output_dim)
            for _ in range(num_experts)
        ])
        
        # Gating network
        self.gate = GatingNetwork(input_dim, num_experts, top_k)
        
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through MoE layer
        
        Returns:
            output: Weighted combination of expert outputs
            load_balance_loss: Auxiliary loss for load balancing
        """
        batch_size, seq_len, input_dim = x.shape
        x_flat = x.view(-1, input_dim)  # [batch_size * seq_len, input_dim]
        
        # Get gating decisions
        gates, expert_indices = self.gate(x_flat)  # [batch_size * seq_len, top_k]
        
        # Initialize output tensor
        output = torch.zeros_like(x_flat[:, :self.experts[0].fc2.out_features])
        
        # Process each expert
        for i in range(self.num_experts):
            # Find tokens routed to this expert
            expert_mask = (expert_indices == i).any(dim=-1)
            
            if expert_mask.sum() == 0:
                continue
                
            # Get tokens for this expert
            expert_input = x_flat[expert_mask]
            expert_output = self.experts[i](expert_input)
            
            # Get corresponding gates
            expert_gate_positions = (expert_indices == i).nonzero()
            expert_gates = gates[expert_gate_positions[:, 0], expert_gate_positions[:, 1]]
            
            # Weighted addition to output
            output[expert_mask] += expert_gates.unsqueeze(-1) * expert_output
        
        # Calculate load balancing loss
        load_balance_loss = self._calculate_load_balance_loss(gates, expert_indices)
        
        # Reshape output
        output = output.view(batch_size, seq_len, -1)
        
        return output, load_balance_loss
    
    def _calculate_load_balance_loss(
        self, 
        gates: torch.Tensor, 
        expert_indices: torch.Tensor
    ) -> torch.Tensor:
        """Calculate auxiliary loss to encourage load balancing"""
        # Count tokens per expert
        expert_counts = torch.zeros(self.num_experts, device=gates.device)
        for i in range(self.num_experts):
            expert_counts[i] = (expert_indices == i).sum().float()
        
        # Calculate fraction of tokens per expert
        total_tokens = expert_counts.sum()
        expert_fractions = expert_counts / (total_tokens + 1e-8)
        
        # Target uniform distribution
        target_fraction = 1.0 / self.num_experts
        
        # L2 loss from uniform distribution
        load_balance_loss = torch.sum((expert_fractions - target_fraction) ** 2)
        
        return self.load_balance_weight * load_balance_loss


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MoEEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(MoEEncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = MixtureOfExperts(
            input_dim=d_model,
            hidden_dim=d_ff,
            output_dim=d_model,
            num_experts=8,
            top_k=2,
            load_balance_weight=0.01
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output, load_balance_loss = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x, load_balance_loss


class MoEDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(MoEDecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = MixtureOfExperts(
            input_dim=d_model,
            hidden_dim=d_ff,
            output_dim=d_model,
            num_experts=8,
            top_k=2,
            load_balance_weight=0.01
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output, load_balance_loss = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x, load_balance_loss
    

class MoETransformer(nn.Module):
    def __init__(self, src_dim, tgt_dim, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(MoETransformer, self).__init__()
        
        # Linear projections instead of embeddings for continuous data
        self.src_projection = nn.Linear(src_dim, d_model)
        self.tgt_projection = nn.Linear(tgt_dim, d_model)
        self.output_projection = nn.Linear(d_model, tgt_dim)
        
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        
        self.encoder_layers = nn.ModuleList([MoEEncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([MoEDecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        
        self.dropout = nn.Dropout(dropout)
        
    def create_padding_mask(self, x):
        return (x.sum(dim=-1) == 0)
    
    def create_look_ahead_mask(self, size, device):
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
        return mask.bool()
    
    def forward(self, src, tgt):
        device = src.device
        batch_size = src.size(0)
        
        # Create padding masks
        src_padding_mask = self.create_padding_mask(src)  # [batch, src_len]
        tgt_padding_mask = self.create_padding_mask(tgt)  # [batch, tgt_len]
        
        # Create look-ahead mask
        tgt_seq_len = tgt.size(1)
        tgt_look_ahead_mask = self.create_look_ahead_mask(tgt_seq_len, device)  # [tgt_len, tgt_len]
        
        # Project to model dimension
        src_embedded = self.dropout(self.positional_encoding(self.src_projection(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.tgt_projection(tgt)))

        # Encoder
        enc_output = src_embedded
        total_load_balance_loss = 0.0
        for enc_layer in self.encoder_layers:
            # Convert padding mask to attention mask format [batch, 1, 1, src_len]
            src_mask = src_padding_mask.unsqueeze(1).unsqueeze(2) if src_padding_mask.any() else None
            enc_output, load_balance_loss = enc_layer(enc_output, src_mask)
            total_load_balance_loss += load_balance_loss
        
        # Decoder
        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            # Source mask for cross-attention
            src_mask = src_padding_mask.unsqueeze(1).unsqueeze(2) if src_padding_mask.any() else None
            
            # Target mask combining padding and look-ahead
            tgt_mask = tgt_look_ahead_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, tgt_len, tgt_len]
            if tgt_padding_mask.any():
                padding_mask = tgt_padding_mask.unsqueeze(1).unsqueeze(2)  # [batch, 1, 1, tgt_len]
                tgt_mask = tgt_mask | padding_mask
            
            dec_output, load_balance_loss = dec_layer(dec_output, enc_output, src_mask, tgt_mask)
            total_load_balance_loss += load_balance_loss
        
        # Project back to target dimension
        output = self.output_projection(dec_output)
        return output, total_load_balance_loss

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

In [ ]:
from torchmetrics import MeanSquaredError, PearsonCorrCoef


def val_transformer_model(model, val_loader, device, criterion):
    model.eval()
    total_loss = 0
    total_raw = 0
    with torch.no_grad():
        for ecog, mspec in val_loader:
            ecog, mspec = ecog.to(device), mspec.to(device)

            mspec_in = torch.zeros_like(mspec)
            mspec_in[:, 1:, :] = mspec[:, :-1, :]

            output, _ = model(ecog, mspec_in) 
            loss = criterion(output, mspec) 
            total_loss += loss.item()

    avg_loss = total_loss / len(val_loader)
    return avg_loss


def train_transformer_model(model, train_loader, val_loader, device, epochs, optimizer, criterion, scheduler, early_stopping):
    train_losses, val_losses = [], []

    for epoch in tqdm(range(epochs), desc="Training"):
        model.train()
        epoch_loss = 0.0
        epoch_balance_loss = 0.0

        for idx, (ecog, mspec) in enumerate(train_loader):
            ecog, mspec = ecog.to(device), mspec.to(device)

            optimizer.zero_grad()

            mspec_in = torch.zeros_like(mspec)
            mspec_in[:, 1:, :] = mspec[:, :-1, :]

            output, load_balance_loss = model(ecog, mspec_in)
            
            primary_loss = criterion(output, mspec)
            
            total_balance_loss = sum(load_balance_loss) if isinstance(load_balance_loss, list) else load_balance_loss
            total_loss = primary_loss + total_balance_loss

            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += primary_loss.item()
            epoch_balance_loss += total_balance_loss.item()

        train_loss = epoch_loss / len(train_loader)
        avg_balance_loss = epoch_balance_loss / len(train_loader)
        val_loss = val_transformer_model(model, val_loader, device, criterion)

        scheduler.step(val_loss)
        early_stopping(val_loss, model)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if (epoch + 1) % 30 == 0 or epoch == 0:
            print(f"\nEpoch {epoch+1}/{epochs}")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Balance Loss : {avg_balance_loss:.4f}")
            print(f"  Val Loss     : {val_loss:.4f}")
            print("-" * 50)

        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break

    if early_stopping.best_model_state is not None:
        model.load_state_dict(early_stopping.best_model_state)
    else:
        print("Warning: No best model found, using last epoch weights.")
    return model, train_losses, val_losses


def test_transformer_model(model, test_loader, device):
    model.eval()
    gen_mspec, target_mspec = [], []

    with torch.no_grad():
        for ecog, mspec in test_loader:
            ecog, mspec = ecog.to(device), mspec.to(device)

            mspec_in = torch.zeros_like(mspec)
            mspec_in[:, 1:, :] = mspec[:, :-1, :]

            output, _ = model(ecog, mspec_in)

            gen_mspec.append(output.cpu())
            target_mspec.append(mspec.cpu())
    
    gen_mspec = torch.cat(gen_mspec)
    target_mspec = torch.cat(target_mspec)

    return gen_mspec, target_mspec

In [ ]:
class MultiScaleSpectralLoss(nn.Module):
    def __init__(self, n_ffts=[1024, 2048, 512], hop_lengths=None, win_lengths=None):
        super().__init__()
        self.n_ffts = n_ffts
        self.hop_lengths = hop_lengths or [n // 4 for n in n_ffts]
        self.win_lengths = win_lengths or n_ffts
        
    def stft_loss(self, pred, target, n_fft, hop_length, win_length):
        # Convert mel-spec back to linear for STFT
        pred_stft = torch.stft(pred.flatten(1), n_fft=n_fft, hop_length=hop_length, 
                              win_length=win_length, return_complex=True)
        target_stft = torch.stft(target.flatten(1), n_fft=n_fft, hop_length=hop_length,
                               win_length=win_length, return_complex=True)
        
        # Magnitude loss
        pred_mag = torch.abs(pred_stft)
        target_mag = torch.abs(target_stft)
        mag_loss = F.l1_loss(pred_mag, target_mag)
        
        # Phase loss
        pred_phase = torch.angle(pred_stft)
        target_phase = torch.angle(target_stft)
        phase_loss = F.l1_loss(pred_phase, target_phase)
        
        return mag_loss + 0.1 * phase_loss
    
    def forward(self, pred, target):
        total_loss = 0
        for n_fft, hop_length, win_length in zip(self.n_ffts, self.hop_lengths, self.win_lengths):
            total_loss += self.stft_loss(pred, target, n_fft, hop_length, win_length)
        return total_loss / len(self.n_ffts)

In [ ]:
class CombinedLoss(torch.nn.Module):
    def __init__(self, w_l1=1.0, w_mse=1.0, w_spec=1.0, w_mel=1.0):
        super().__init__()

        self.w_l1 = w_l1
        self.w_mse = w_mse
        self.w_spec = w_spec
        self.w_mel = w_mel

        self.l1_loss = torch.nn.L1Loss()
        self.mse_loss = torch.nn.MSELoss()
        self.spectral_loss = MultiScaleSpectralLoss()

        self.last_losses = {}

    def forward(self, pred, target):
        l1 = self.l1_loss(pred, target)
        mse = self.mse_loss(pred, target)
        spectral = self.spectral_loss(pred, target)

        mel_diff = torch.mean(
            torch.abs(pred - target) * torch.log(1 + torch.abs(target))
        )

        # Save raw magnitudes
        self.last_losses = {
            "l1": l1.item(),
            "mse": mse.item(),
            "spectral": spectral.item(),
            "mel": mel_diff.item()
        }

        total = (
            self.w_l1 * l1 +
            self.w_mse * mse +
            self.w_spec * spectral +
            self.w_mel * mel_diff
        )

        return total

### 8. From mel-spec to audio to text

In [ ]:
def mspec2audio2text_whisper(mspec, hifigan, processor, model):
    device = hifigan.device

    # --- Mel to tensor ---
    if not isinstance(mspec, torch.Tensor):
        mspec = torch.from_numpy(mspec)

    mspec = mspec.float()

    # Ensure shape: (B, 80, T)
    if mspec.ndim == 2:
        if mspec.shape[0] == 80:
            mspec = mspec.unsqueeze(0)
        elif mspec.shape[1] == 80:
            mspec = mspec.transpose(0, 1).unsqueeze(0)

    mspec = mspec.to(device)

    # --- HiFi-GAN vocoding ---
    with torch.no_grad():
        audio = hifigan.decode_batch(mspec)

    audio = audio.squeeze(0)
    audio = audio / (audio.abs().max() + 1e-8)

    # HiFi-GAN output is 22050 Hz → Whisper needs 16000 Hz
    audio_16k = librosa.resample(
        audio.cpu().numpy(),
        orig_sr=22050,
        target_sr=16000
    )

    audio_16k = audio_16k / (np.max(np.abs(audio_16k)) + 1e-8)

    # --- Whisper processing ---
    inputs = processor(
        audio_16k,
        sampling_rate=16000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            inputs.input_features,
            language="en",        # remove if multilingual
            task="transcribe"
        )

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]
    transcription = np.char.lower(transcription)

    return audio_16k, transcription

### 9. Plot mel-spectrogram

In [ ]:
def plot_mspec(ori_mspec, gen_mspec, title=None, save_path=None):
    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    im0 = axs[0].imshow(ori_mspec.T, aspect='auto', origin='lower', interpolation='none', cmap='magma')
    axs[0].set_title('Original Mel-spectrogram')
    axs[0].set_xlabel('Frame')
    axs[0].set_ylabel('Mel Bin')
    fig.colorbar(im0, ax=axs[0], format='%+2.0f dB')
    
    im1 = axs[1].imshow(gen_mspec.T, aspect='auto', origin='lower', interpolation='none', cmap='magma')
    axs[1].set_title('Generated Mel-spectrogram')
    axs[1].set_xlabel('Frame')
    axs[1].set_ylabel('Mel Bin')
    fig.colorbar(im1, ax=axs[1], format='%+2.0f dB')
    
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300)
    plt.show()

# Prepare dataset

### 1. Load data

In [ ]:
dictionary = load_data('/home/mega/Desktop/Phuong/Brain2Speech/t12.2022.05.03_fiftyWordSet.mat')
ecog, mspec, label, text = dictionary['ecog'], dictionary['mspec'], dictionary['label'], dictionary['text']

### 2. Align ecog with mel-spec

In [ ]:
aligned_mspec = [mspec[i][align_from_distances(ecog[i], mspec[i], debug=False)] 
                 for i in tqdm(range(len(mspec)), desc="Aligning ECoG-Mel-spec")]
aligned_mspec = np.array(aligned_mspec, dtype=np.float32)
print(aligned_mspec.shape)
print(ecog.shape)

### 3. Split data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, label_train, label_val, label_test = prepare_dataloader_custom(
    ecog, aligned_mspec, label,
    test_size=0.1, val_size=0.1,
    exclude_labels=None
)

print(f"\nData split summary:")
print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

In [ ]:
test_texts = [text[np.where(label == l)[0][0]] for l in label_test]
test_texts = np.char.lower(test_texts)

### 4. Data augmentaion using diffusion model

In [ ]:
import matplotlib.pyplot as plt

num_epochs = 100
batch_size = 4
learning_rate = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Train with linear schedule
print("Training diffusion model with LINEAR schedule...")
model_lin, diff_lin, loss_lin = train_diffusion_model(
    X_train,
    label_train,
    num_epochs=num_epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    device=device,
    schedule="linear",
)

# Train with cosine schedule
print("\nTraining diffusion model with COSINE schedule...")
model_cos, diff_cos, loss_cos = train_diffusion_model(
    X_train,
    label_train,
    num_epochs=num_epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    device=device,
    schedule="cosine",
)

In [ ]:
# Plot SNR curves
t = torch.arange(diff_lin.timesteps, device=diff_lin.betas.device)

plt.figure(figsize=(8, 4))
plt.plot(t.cpu(), diff_lin.snr.cpu(), label="linear schedule", linewidth=2)
plt.plot(t.cpu(), diff_cos.snr.cpu(), label="cosine schedule", linewidth=2)
plt.yscale("log")
plt.xlabel("timestep")
plt.ylabel("SNR")
plt.title("SNR vs timestep: linear vs cosine")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot loss curves
plt.figure(figsize=(8, 4))
plt.plot(range(num_epochs), loss_lin, label="linear schedule", linewidth=2)
plt.plot(range(num_epochs), loss_cos, label="cosine schedule", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Diffusion training loss: linear vs cosine schedule")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
diffusion_ratio = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_samples_per_label = int(round((len(X_train) / len(np.unique(label_train))) * diffusion_ratio))
print(f"\nGenerating {num_samples_per_label} samples per label for diffusion augmentation")

In [ ]:
print("\nGenerating synthetic ECoG (linear schedule)...")
synthetic_ecog_lin, synthetic_labels_lin = generate_synthetic_ecog(
    model_lin, diff_lin, label_train, num_samples_per_label=num_samples_per_label
)

In [ ]:
print("\nGenerating synthetic ECoG (cosine schedule)...")
synthetic_ecog_cos, synthetic_labels_cos = generate_synthetic_ecog(
    model_cos, diff_cos, label_train, num_samples_per_label=num_samples_per_label
)

In [ ]:
print(f"\nOriginal train set: {X_train.shape}")
print(f"Linear synthetic:   {synthetic_ecog_lin.shape}")
print(f"Cosine synthetic:   {synthetic_ecog_cos.shape}")

In [ ]:
def build_synthetic_mspec(synthetic_labels):
    synthetic_mspec = []
    for syn_label in synthetic_labels:
        matching_indices = np.where(label_train == syn_label)[0]
        if len(matching_indices) > 0:
            idx = np.random.choice(matching_indices)
            synthetic_mspec.append(y_train[idx])
        else:
            synthetic_mspec.append(y_train[0])
    return np.array(synthetic_mspec, dtype=np.float32)

synthetic_mspec_lin = build_synthetic_mspec(synthetic_labels_lin)
synthetic_mspec_cos = build_synthetic_mspec(synthetic_labels_cos)

In [ ]:
aug_ecog_lin = np.concatenate((X_train, synthetic_ecog_lin), axis=0)
aug_mspec_lin = np.concatenate((y_train, synthetic_mspec_lin), axis=0)

aug_ecog_cos = np.concatenate((X_train, synthetic_ecog_cos), axis=0)
aug_mspec_cos = np.concatenate((y_train, synthetic_mspec_cos), axis=0)

print("\nAugmented dataset shapes:")
print(f"Linear  : ECoG {aug_ecog_lin.shape}, Mel {aug_mspec_lin.shape}")
print(f"Cosine  : ECoG {aug_ecog_cos.shape}, Mel {aug_mspec_cos.shape}")

### 5. Feature extraction

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch, os

batch_size = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SAFETY SETTINGS
num_workers = 0                    
pin_memory = (device.type == "cuda")
persistent_workers = False          

feature_state = True

if feature_state:
    feature_extractor = ConvLSTMFeatureExtractor(
        input_size=256,
        conv_channels=64,
        lstm_hidden=128,
        output_size=80
    ).to(device)

    feature_extractor.eval()
    print('\nExtracting features from augmented ECoG data...')

    with torch.no_grad():
        train_features = feature_extractor(
            torch.as_tensor(aug_ecog_lin, dtype=torch.float32, device=device)
        ).cpu()

        val_features = feature_extractor(
            torch.as_tensor(X_val, dtype=torch.float32, device=device)
        ).cpu()

        test_features = feature_extractor(
            torch.as_tensor(X_test, dtype=torch.float32, device=device)
        ).cpu()

    train_dataset = TensorDataset(
        train_features,
        torch.as_tensor(aug_mspec_lin, dtype=torch.float32)
    )

    val_dataset = TensorDataset(
        val_features,
        torch.as_tensor(y_val, dtype=torch.float32)
    )

    test_dataset = TensorDataset(
        test_features,
        torch.as_tensor(y_test, dtype=torch.float32)
    )

else:
    train_dataset = TensorDataset(
        torch.as_tensor(aug_ecog_lin, dtype=torch.float32),
        torch.as_tensor(aug_mspec_lin, dtype=torch.float32)
    )

    val_dataset = TensorDataset(
        torch.as_tensor(X_val, dtype=torch.float32),
        torch.as_tensor(y_val, dtype=torch.float32)
    )

    test_dataset = TensorDataset(
        torch.as_tensor(X_test, dtype=torch.float32),
        torch.as_tensor(y_test, dtype=torch.float32)
    )


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print('\nFinal dataset sizes:')
print(f'Training samples: {len(train_loader.dataset)}')
print(f'Validation samples: {len(val_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')

### 6. Train MoETransformer

In [ ]:
def z_norm(m):
    return (m - m.mean()) / (m.std() + 1e-6)

from torchmetrics import PearsonCorrCoef

def framewise_pcc(gen, tgt):
    # gen, tgt: [B, M, T]
    metric = PearsonCorrCoef().to(gen.device)
    B, M, T = gen.shape
    pccs = []

    for b in range(B):
        g = z_norm(gen[b])
        t = z_norm(tgt[b])
        for i in range(T):
            pccs.append(metric(g[:, i], t[:, i]))

    return torch.mean(torch.stack(pccs))

In [ ]:
import re

def clean_words(word_list):
    cleaned = []

    for w in word_list:
        # Convert bytes → str
        if isinstance(w, bytes):
            w = w.decode("utf-8", errors="ignore")

        # Safety: force str
        w = str(w)

        # Remove punctuation
        w = re.sub(r"[?.!,]", "", w)

        # Normalize spacing
        w = w.strip()
        
        if len(w) > 0:
            w = w.lower()
        
        cleaned.append(w)
    
    return cleaned

In [ ]:
# TRANSFORMER
import os
import random
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from speechbrain.inference.vocoders import HIFIGAN
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from torchmetrics.text import CharErrorRate, WordErrorRate
from torchmetrics.regression import MeanSquaredError
from pesq import pesq
from pystoi.stoi import stoi

output_dir = os.path.join(os.getcwd(), "outputs_transformer")
os.makedirs(output_dir, exist_ok=True)

# =========================
# Reproducibility
# =========================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    
# =========================
# Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sr = 16000


# =========================
# Models
# =========================
hifigan = HIFIGAN.from_hparams(
    source="speechbrain/tts-hifigan-ljspeech",
    savedir="pretrained_models/tts-hifigan-ljspeech"
).eval()

processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
asr_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-medium"
).eval()


# =========================
# Seeds
# =========================
seeds = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021]
results = []


# =========================
# Main Loop
# =========================
for seed in seeds:
    print(f"\n===== Running seed {seed} =====")
    set_seed(seed)

    # --- Initialize Transformer ---
    if feature_state:
        transformer_model = MoETransformer(
            src_dim=80, tgt_dim=80, d_model=512,
            num_heads=8, num_layers=6,
            d_ff=2048, max_seq_length=101, dropout=0.1
        ).to(device)
    else:
        transformer_model = MoETransformer(
            src_dim=256, tgt_dim=80, d_model=512,
            num_heads=8, num_layers=6,
            d_ff=2048, max_seq_length=101, dropout=0.1
        ).to(device)

    criterion = CombinedLoss(w_l1=1.0, w_mse=1.0, w_spec=0.1, w_mel=0.5).to(device)

    optimizer = torch.optim.AdamW(
        list(transformer_model.parameters()) + list(criterion.parameters()),
        lr=1e-4, weight_decay=1e-5
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    early_stopping = EarlyStopping(patience=10, delta=0.01)

    model_path = os.path.join(output_dir, f"transformer_model_seed{seed}.pth")

    # =========================
    # Train or Load
    # =========================
    if os.path.exists(model_path):
        print("Loading existing model for this seed...")
        transformer_model.load_state_dict(
            torch.load(model_path, map_location=device)
        )
    else:
        print("Training model...")
        transformer_model, train_losses, val_losses = train_transformer_model(
            transformer_model,
            train_loader,
            val_loader,
            device,
            epochs=150,
            optimizer=optimizer,
            criterion=criterion,
            scheduler=scheduler,
            early_stopping=early_stopping
        )
        torch.save(transformer_model.state_dict(), model_path)

    transformer_model.eval()   

    # =========================
    # Test
    # =========================
    gen_mspec, target_mspec = test_transformer_model(
        transformer_model, test_loader, device
    )

    mse = MeanSquaredError()(gen_mspec, target_mspec).item()
    rmse = np.sqrt(mse)
    pcc = framewise_pcc(gen_mspec, target_mspec).item()

    gen_texts, target_texts = [], []
    pesq_scores, stoi_scores = [], []

    for i in tqdm(range(len(gen_mspec))):
        gen_audio, gen_text = mspec2audio2text_whisper(gen_mspec[i], hifigan, processor, asr_model)
        target_audio, target_text = mspec2audio2text_whisper(target_mspec[i], hifigan, processor, asr_model)

        gen_audio = np.asarray(gen_audio, dtype=np.float32)
        target_audio = np.asarray(target_audio, dtype=np.float32)
        gen_texts.append(gen_text)
        gen_texts = clean_words(gen_texts)
        target_texts.append(target_text)
        target_texts = clean_words(target_texts)

        # Metrics
        pesq_score = pesq(sr, target_audio.reshape(-1), gen_audio.reshape(-1), 'wb')
        pesq_scores.append(pesq_score)

        stoi_score = stoi(target_audio.reshape(-1), gen_audio.reshape(-1), sr, extended=False)
        stoi_scores.append(stoi_score)

    # Compute metrics
    cer = CharErrorRate()(gen_texts, test_texts).item()
    wer = WordErrorRate()(gen_texts, test_texts).item()
    avg_pesq = float(np.mean(pesq_scores))
    avg_stoi = float(np.mean(stoi_scores))

    print(f"Seed {seed} Results:")
    print(f"  RMSE: {rmse:.4f} | PCC: {pcc:.4f}")
    print(f"  CER: {cer*100:.4f}% | WER: {wer*100:.4f}%")
    print(f"  PESQ: {avg_pesq:.4f} | STOI: {avg_stoi:.4f}")
    
    # Save results
    results.append({
        "seed": seed,
        "RMSE": float(f"{rmse:.4f}"),
        "PCC": float(f"{pcc:.4f}"),
        "CER": float(f"{cer*100:.4f}"),
        "WER": float(f"{wer*100:.4f}"),
        "PESQ": float(f"{avg_pesq:.4f}"),
        "STOI": float(f"{avg_stoi:.4f}")
    })

# Save all results to a file
results_path = os.path.join(output_dir, "transformer_results_10seeds.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print(f"\nAll done! Results saved to: {results_path}")